In [1]:
import numpy as np
import pandas as pd

# 視覺化/模型等會用到的套件
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV

# 讀取 College 資料集
# 確認 College.csv 跟這個 .py 在同一資料夾
college = pd.read_csv("College.csv", index_col=0)  # index_col=0 把第一欄學校名字當 index

# 目標變數：Apps
y = college["Apps"]

# 解釋變數：其他欄位
X = college.drop(columns=["Apps"])

# 'Private' 是類別變數，要轉成 0/1 dummy
X = pd.get_dummies(X, drop_first=True)  # drop_first=True 避免 dummy trap

print("X 形狀:", X.shape)
print("y 形狀:", y.shape)
print("X 前幾列：")
print(X.head())


X 形狀: (777, 17)
y 形狀: (777,)
X 前幾列：
                              Accept  Enroll  Top10perc  Top25perc  \
Abilene Christian University    1232     721         23         52   
Adelphi University              1924     512         16         29   
Adrian College                  1097     336         22         50   
Agnes Scott College              349     137         60         89   
Alaska Pacific University        146      55         16         44   

                              F.Undergrad  P.Undergrad  Outstate  Room.Board  \
Abilene Christian University         2885          537      7440        3300   
Adelphi University                   2683         1227     12280        6450   
Adrian College                       1036           99     11250        3750   
Agnes Scott College                   510           63     12960        5450   
Alaska Pacific University             249          869      7560        4120   

                              Books  Personal  PhD  Terminal  

### (a)

Split the data set into a training set and a test set.
Use `Apps` as the response and the remaining variables as predictors.


In [2]:
# ========== (a) 切分訓練集 / 測試集 ==========

# 這裡用 70% 當訓練集、30% 當測試集，可以依照作業需求調整
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,      # 30% 當 test
    random_state=0      # 固定 random_state，方便重現
)

print("訓練集大小:", X_train.shape[0])
print("測試集大小:", X_test.shape[0])


訓練集大小: 543
測試集大小: 234


### (b)

Fit a linear model using least squares on the training set.
Report the test error obtained (e.g., test MSE).


In [3]:
# ========== (b) 線性回歸 (Least Squares) ==========

linreg = LinearRegression()

# 用訓練集估參數
linreg.fit(X_train, y_train)

# 在測試集上做預測
y_pred_lin = linreg.predict(X_test)

# 計算測試誤差（這裡用 MSE）
mse_lin = mean_squared_error(y_test, y_pred_lin)
print("\n(b) 線性回歸 Test MSE:", mse_lin)



(b) 線性回歸 Test MSE: 1659682.1719133682


### (c)

Fit a ridge regression model on the training set, with the tuning parameter
\(\lambda\) chosen by cross-validation. Report the test error obtained.


In [4]:
# ========== (c) Ridge Regression (λ by CV) ==========

# Ridge/Lasso 都需要標準化，所以用 pipeline：StandardScaler + RidgeCV
alphas = np.logspace(-3, 5, 50)  # 一組候選 λ（10^-3 到 10^5）

ridge_model = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=alphas, cv=10)    # 10-fold CV 選最佳 lambda
)

ridge_model.fit(X_train, y_train)

# 從 pipeline 裡拿出真正的 RidgeCV 物件
ridge_cv = ridge_model.named_steps["ridgecv"]

print("\n(c) Ridge Regression")
print("選到的最佳 lambda (alpha):", ridge_cv.alpha_)

# 在測試集上預測
y_pred_ridge = ridge_model.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
print("Ridge Test MSE:", mse_ridge)



(c) Ridge Regression
選到的最佳 lambda (alpha): 1.8420699693267144
Ridge Test MSE: 1746102.978165594


### (d)

Fit a lasso model on the training set, with the tuning parameter
\(\lambda\) chosen by cross-validation.
Report the test error obtained and the number of non-zero coefficient estimates.


In [5]:
# ========== (d) Lasso (λ by CV) ==========

lasso_model = make_pipeline(
    StandardScaler(),
    LassoCV(cv=10, random_state=0)  # 10-fold CV
)

lasso_model.fit(X_train, y_train)

lasso_cv = lasso_model.named_steps["lassocv"]

print("\n(d) Lasso Regression")
print("選到的最佳 lambda (alpha):", lasso_cv.alpha_)

# 測試集預測
y_pred_lasso = lasso_model.predict(X_test)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
print("Lasso Test MSE:", mse_lasso)

# 數一下有幾個係數 ≠ 0
coef = lasso_cv.coef_
n_nonzero = np.sum(coef != 0)
print("Lasso 非零係數個數:", n_nonzero)



(d) Lasso Regression
選到的最佳 lambda (alpha): 10.593988724528039
Lasso Test MSE: 1752352.942985331
Lasso 非零係數個數: 14


### (e)

Fit a principal components regression (PCR) model on the training set.
Choose the number of components \(M\) by cross-validation.
Report the test error obtained and the value of \(M\) selected by cross-validation.


In [6]:
# ========== (e) PCR: PCA + LinearRegression (M by CV) ==========

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# 建 PCR pipeline: StandardScaler -> PCA -> LinearRegression
pcr_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA()),
    ("linreg", LinearRegression())
])

# 我們要在 1 ~ 最大特徵數 之間找最佳的主成分數 M
max_components = X_train.shape[1]

param_grid = {
    "pca__n_components": list(range(1, max_components + 1))
}

pcr_cv = GridSearchCV(
    pcr_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",  # 越大越好（因為是負的 MSE）
    cv=10,
    n_jobs=-1
)

pcr_cv.fit(X_train, y_train)

best_M = pcr_cv.best_params_["pca__n_components"]
print("\n(e) PCR (Principal Components Regression)")
print("CV 選到的最佳主成分數 M:", best_M)

# 用最佳 M 的模型對 test set 做預測
best_pcr_model = pcr_cv.best_estimator_
y_pred_pcr = best_pcr_model.predict(X_test)
mse_pcr = mean_squared_error(y_test, y_pred_pcr)
print("PCR Test MSE:", mse_pcr)



(e) PCR (Principal Components Regression)
CV 選到的最佳主成分數 M: 17
PCR Test MSE: 1659682.171913369
